In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/nolabel_test_dataset.tsv
/kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/test_dataset.tsv
/kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/train_dataset.tsv
/kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/valid_dataset.tsv


In [2]:
!git clone https://github.com/linwhitehat/ET-BERT.git
%cd ET-BERT

Cloning into 'ET-BERT'...
remote: Enumerating objects: 791, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 791 (delta 304), reused 300 (delta 300), pack-reused 408 (from 1)
Receiving objects: 100% (791/791), 13.38 MiB | 32.25 MiB/s, done.
Resolving deltas: 100% (414/414), done.
/kaggle/working/ET-BERT


In [3]:
!pip install -r requirements.txt
!pip install torch torchvision torchaudio transformers pandas numpy scikit-learn

In [4]:
import os
import torch
from transformers import BertTokenizer, BertModel
print("Current Working Directory:", os.getcwd())
print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

Current Working Directory: /kaggle/working/ET-BERT
PyTorch Version: 2.11.0+cu128
GPU Available: True


In [5]:
!dir

bert_base_config.json  datasets     inference  preprocess.py  requirements.txt
corpora		       fine-tuning  LICENSE    pre-training   uer
data_process	       images	    models     README.md      vocab_process


In [6]:
!cat README.md

# ET-BERT: A Contextualized Datagram Representation with Pre-training Transformers for Encrypted Traffic Classification

<!-- 
[![codebeat badge](https://codebeat.co/badges/f75fab90-6d00-44b4-bb42-d19067400243)](https://codebeat.co/projects/github-com-linwhitehat-et-bert-main) 
![](https://img.shields.io/badge/license-MIT-000000.svg) 
[![arXiv](https://img.shields.io/badge/arXiv-1909.05658-<color>.svg)](https://arxiv.org/abs/2202.06335) 
-->

<p align="center">
  <a href=''><img src='https://img.shields.io/badge/License-MIT-brightgreen'></a> 
  <a href='https://arxiv.org/abs/2202.06335'><img src='https://img.shields.io/badge/arXiv-1909.05658-<color>.svg'></a> 
  <a href='https://dl.acm.org/doi/10.1145/3485447.3512217' target='_blank'><img src="https://img.shields.io/badge/WWW'22-Paper-blue"></a>
</p>

**Note:**
- ⭐ **Please leave a <font color='orange'>STAR</font> if you like this project!** ⭐
- If you find any <font color='red'>incorrect</font> / <font color='red'>inappropriate</font>

In [7]:
import os

if not os.getcwd().endswith('ET-BERT'):
    try:
        os.chdir('ET-BERT')
    except FileNotFoundError:
        pass

print("Current Directory:", os.getcwd())

Current Directory: /kaggle/working/ET-BERT


In [8]:
import gdown
import os

os.makedirs('models', exist_ok=True)

file_id = '1r1yE34dU2W8zSqx1FkB8gCWri4DQWVtE'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'models/pretrained_model.bin'

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1r1yE34dU2W8zSqx1FkB8gCWri4DQWVtE
From (redirected): https://drive.google.com/uc?id=1r1yE34dU2W8zSqx1FkB8gCWri4DQWVtE&confirm=t&uuid=617c8f6d-86ac-45cc-ab2b-c6a07f2a6fbe
To: /kaggle/working/ET-BERT/models/pretrained_model.bin
100%|██████████| 716M/716M [00:03<00:00, 238MB/s]


'models/pretrained_model.bin'

In [9]:
import os
os.environ['PYTHONPATH'] = os.getcwd()
print("PYTHONPATH set to:", os.environ['PYTHONPATH'])

PYTHONPATH set to: /kaggle/working/ET-BERT


In [10]:
import os

dataset_path = "datasets"

if os.path.exists(dataset_path):
    print(f"Contents of '{dataset_path}':")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
else:
    print(f"Directory '{dataset_path}' does not exist.")

Contents of 'datasets':
datasets/
    README.md
    CSTNET-TLS 1.3/
        readme.md


In [11]:
import os

readme_path_1 = "datasets/README.md"
readme_path_2 = "datasets/CSTNET-TLS 1.3/readme.md"

def read_info(path):
    if os.path.exists(path):
        print(f"--- {path} ---")
        with open(path, "r", encoding="utf-8") as file:
            print(file.read())
        print("\n")
    else:
        print(f"File not found: {path}")

read_info(readme_path_1)
read_info(readme_path_2)

--- datasets/README.md ---
### Shared Datasets
We generate the training data (packet-level) from the public datasets `ISCX-VPN` and `USTC-TFC` selected in the paper and share them to the cloud drive for testing purposes.

1. [ISCX-VPN](https://drive.google.com/drive/folders/1is609sosAdqf9YJAfwr72hBqM4OeNuZq?usp=sharing)
2. [USTC-TFC](https://drive.google.com/file/d/1F09zxln9iFg2HWoqc6m4LKFhYK7cDQv_/view?usp=sharing)
3. [Others](https://drive.google.com/drive/folders/1is609sosAdqf9YJAfwr72hBqM4OeNuZq?usp=sharing)

### Labels of Datasets

![image](https://user-images.githubusercontent.com/20349381/167751578-655a1e5c-c45d-4fda-8324-5c95a5e1d476.png)



--- datasets/CSTNET-TLS 1.3/readme.md ---
******
data download link
******

[cstnet-tls 1.3 dataset](https://drive.google.com/drive/folders/1JSsYmevkxQFanoKOi_i1ooA6pH3s9sDr?usp=sharing)

[fine-tuning cstnet-tls 1.3](https://drive.google.com/drive/folders/1KlZatGoNm-4qu04z0LfrTpZr2oDaHfzr?usp=sharing)

**Labels of Dataset**
![image](https:/

In [12]:
import os

# Create the required folders
output_dir = "datasets/cstnet-tls1.3/packet"
os.makedirs(output_dir, exist_ok=True)

print(f"Directory created at: {output_dir}")
print("Downloading dataset from Google Drive...")

# Download the Google Drive folder using gdown
!gdown --folder https://drive.google.com/drive/folders/1KlZatGoNm-4quO4z0LfrTpZr2oDaHfzr -O datasets/cstnet-tls1.3/packet/

Directory created at: datasets/cstnet-tls1.3/packet
Retrieving folder contents
Failed to retrieve folder contents


In [13]:
!sed -i 's|/data2/lxj/pre-train/results/confusion_matrix|./confusion_matrix.txt|g' fine-tuning/run_classifier.py

In [14]:
import os
import torch
import gc

# Clear residual cache
gc.collect()
torch.cuda.empty_cache()

os.environ['PYTHONPATH'] = os.getcwd()

!python fine-tuning/run_classifier.py \
    --pretrained_model_path models/pretrained_model.bin \
    --vocab_path models/encryptd_vocab.txt \
    --train_path /kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/train_dataset.tsv \
    --dev_path /kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/valid_dataset.tsv \
    --test_path /kaggle/input/datasets/anikmajumdar01/encrypted-network-traffic-classification-dataset/test_dataset.tsv \
    --epochs_num 3 \
    --batch_size 32 \
    --embedding word_pos_seg \
    --encoder transformer \
    --mask fully_visible \
    --seq_length 128 \
    --learning_rate 2e-5

Batch size:  32
The number of training instances: 465367
2 GPUs are available. Let's use them.
Start training.
  0%|                                                     | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
Epoch id: 1, Training steps: 100, Avg loss: 4.824
Epoch id: 1, Training steps: 200, Avg loss: 4.793
Epoch id: 1, Training steps: 300, Avg loss: 4.766
Epoch id: 1, Training steps: 400, Avg loss: 4.700
Epoch id: 1, Training steps: 500, Avg loss: 4.608
Epoch id: 1, Training steps: 600, Avg loss: 4.536
Epoch id: 1, Training steps: 700, Avg loss: 4.475
Epoch id: 1, Training steps: 800, Avg loss: 4.394
Epoch id: 1, Training steps: 900, Avg loss: 4.314
Epoch id: 1, Training steps: 1000, Avg loss: 4.235
Epoch id: 1, Training steps: 1100, Avg los